# Agrupaciones con `.agg()` — Superstore

Cuatro ejercicios progresivos sobre el mismo dataset.
Cada uno cubre una forma distinta de usar `.agg()`:
lista de funciones → diccionario + MultiIndex → agregaciones nombradas → groupby dos columnas.


In [1]:
import pandas as pd
from pathlib import Path

def find_project_root():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / 'data').exists():
            return parent
    return current

ROOT = find_project_root()
df   = pd.read_csv(ROOT / 'data' / 'external' / 'train.csv')
print(f'Shape: {df.shape}')
print(df.columns.tolist())


def fmt_euro(df, decimales=2):
    # Detecta solo las columnas numericas y les aplica el formato
    # {:,.2f} -> coma como separador de miles, 2 decimales
    # No toca columnas de texto ni indices
    fmt = f'{{:,.{decimales}f}} €'
    numericas = df.select_dtypes(include='number').columns
    return df.style.format({col: fmt for col in numericas})

print('fmt_euro() lista para usar')

Shape: (9800, 18)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']
fmt_euro() lista para usar


---
## Ejercicio 1 — `.agg()` con lista de funciones

Agrupa por `Category` y aplica sobre `Sales` las funciones
`mean`, `sum`, `count`, `min` y `max` en una sola operación.

El resultado debe ser un DataFrame donde el índice son las categorías
y cada columna es una función aplicada sobre `Sales`.

¿Qué categoría tiene el ticket medio más alto? ¿Y cuál tiene más transacciones?


In [2]:
#agrupacion = df.groupby("Category")["Sales"].agg(["mean", "sum", "count", "min", "max"])
#print(agrupacion)
df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


---
## Ejercicio 2 — `.agg()` con diccionario + aplanar MultiIndex

Agrupa por `Region` y aplica funciones distintas a columnas distintas:
- Sobre `Sales`: media y suma
- Sobre `Order ID`: cuántos pedidos únicos hay por región (`nunique`)

El resultado tendrá MultiIndex en las columnas.
Aplánalo para que los nombres queden como `Sales_mean`, `Sales_sum`, `Order ID_nunique`.
Después haz `reset_index()`.

¿Qué región tiene más pedidos únicos pero menor ticket medio?


In [3]:
agrupacion = df.groupby("Region").agg({
    "Sales": ["mean","sum"],
    "Order ID": ["nunique"],
})
print(agrupacion)

              Sales              Order ID
               mean          sum  nunique
Region                                   
Central  216.357889  492646.9132     1156
East     240.401697  669518.7260     1369
South    243.524067  389151.4590      810
West     226.184613  710219.6845     1587


In [4]:
agrupacion = df.groupby("Region").agg({
    "Sales": ["mean", "sum"],
    "Order ID": ["nunique"],
})

# Aplanar MultiIndex antes de acceder a las columnas
agrupacion.columns = ["_".join(col) for col in agrupacion.columns]
agrupacion = agrupacion.reset_index()

print(agrupacion)

reg_pedidos = agrupacion.loc[agrupacion['Order ID_nunique'].idxmax(), 'Region']
print(f'La región con más pedidos es: {reg_pedidos}')


    Region  Sales_mean    Sales_sum  Order ID_nunique
0  Central  216.357889  492646.9132              1156
1     East  240.401697  669518.7260              1369
2    South  243.524067  389151.4590               810
3     West  226.184613  710219.6845              1587
La región con más pedidos es: West


---
## Ejercicio 3 — Agregaciones nombradas

Agrupa por `Sub-Category` y usa la forma de tuplas nombradas
para producir directamente estas columnas con esos nombres exactos:

- `revenue_total` — suma de ventas
- `ticket_medio` — media de ventas
- `num_pedidos` — conteo de transacciones
- `venta_maxima` — máximo de ventas
- `venta_minima` — mínimo de ventas

Ordena por `revenue_total` de mayor a menor.
No debe haber ningún `.rename()` en tu código.


In [5]:
agrupNomb = (
    df.groupby('Sub-Category').agg(
        renueve_total = ('Sales', 'sum'),
        ticket_medio  = ('Sales', 'mean'),
        num_pedidos   = ('Sales', 'count'),
        venta_maxima  = ('Sales', 'max'),
        venta_minima  = ('Sales', 'min')
    )
    .reset_index()
    .sort_values('renueve_total', ascending=False)
)

# num_pedidos es un conteo, no un importe — se excluye del formato euro
# fmt_euro solo toca columnas numericas, pero num_pedidos no deberia
# mostrarse con decimales ni simbolo, asi que lo separamos
fmt_euro(
    agrupNomb.drop(columns='num_pedidos'),
    decimales=2
)

,Sub-Category,renueve_total,ticket_medio,venta_maxima,venta_minima
13,Phones,"327,782.45 €",374.18 €,"4,548.81 €",2.97 €
5,Chairs,"322,822.73 €",531.83 €,"4,416.17 €",26.64 €
14,Storage,"219,343.39 €",263.63 €,"2,934.33 €",4.46 €
16,Tables,"202,810.63 €",645.89 €,"4,297.64 €",24.37 €
3,Binders,"200,028.79 €",134.07 €,"9,892.74 €",0.56 €
11,Machines,"189,238.63 €","1,645.55 €","22,638.48 €",11.56 €
0,Accessories,"164,186.70 €",217.18 €,"3,347.37 €",0.99 €
6,Copiers,"146,248.09 €","2,215.88 €","17,499.95 €",299.99 €
4,Bookcases,"113,813.20 €",503.60 €,"4,404.90 €",35.49 €
1,Appliances,"104,618.40 €",227.93 €,"2,625.12 €",0.44 €


---
## Ejercicio 4 — Groupby por dos columnas

Agrupa por `Region` y `Category` a la vez usando agregaciones nombradas sobre `Sales`:

- `revenue_total`
- `num_transacciones`
- `ticket_medio`



In [6]:
resultado = (
    df
    .groupby(["Region", "Category"])
    .agg(
        renueve_total = ('Sales', 'sum'),
        ticket_medio  = ('Sales', 'mean'),
        num_pedidos   = ('Sales', 'count')
    ).reset_index()#
    .sort_values(["Region", "renueve_total"], ascending=[False, False])
)

print(resultado)

     Region         Category  renueve_total  ticket_medio  num_pedidos
11     West       Technology    247404.9300    420.042326          589
9      West        Furniture    245348.2455    355.062584          691
10     West  Office Supplies    217466.5090    116.917478         1860
8     South       Technology    148195.2080    512.786187          289
7     South  Office Supplies    124424.7710    126.576573          983
6     South        Furniture    116531.4800    357.458528          326
5      East       Technology    263116.5270    499.272347          527
3      East        Furniture    206461.3880    349.342450          591
4      East  Office Supplies    199940.8110    119.940499         1667
2   Central       Technology    168739.2080    413.576490          408
1   Central  Office Supplies    163590.2430    116.933698         1399
0   Central        Furniture    160317.4622    341.100983          470
